# GPU ODE Benchmark Suite

Modular problem library + GPU divergence-reduction strategies.

## Problem families
| ID | Name | Stiffness | State dim |
|----|------|-----------|----------|
| NS | Nonstiff oscillator | None (ω∈[1,4]) | d |
| MS | Mildly stiff decay | λ∈[1,10] | d |
| SS | Strongly stiff linear | λ∈[100,2000] | d |
| MX | Mixed stiffness (fast+slow modes) | κ∈[1,1000] | 2d |
| VZ | Van der Pol (scalar, μ∈[1,50]) | nonlinear stiff | 2 |
| RB | Robertson (3-species) | extreme stiff | 3 |
| LD | Large-dim stiff decay | λ∈[5,500] | 64 |

## GPU balancing strategies
| Strategy | Description |
|----------|-------------|
| `baseline` | Mask inactive (current approach) |
| `compaction` | Active-set compaction each iteration |
| `buckets` | Partition by stiffness regime at start |
| `rebatch` | Periodic compaction every K steps |
| `split_queue` | Separate explicit/implicit queues |

## Instrumentation
Active fraction, step-size spread (CoV), rejection rate, Newton iterations/step, compaction overhead.

In [1]:
#%pip install -q -U "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
import os, time, math, pickle, subprocess, threading
from dataclasses import dataclass, field
from typing import Callable, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

jax.config.update('jax_enable_x64', True)

# Mount Google Drive (Colab only — harmless no-op elsewhere)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not in Colab — Drive checkpointing disabled')

print('JAX version   :', jax.__version__)
print('Devices       :', jax.devices())
print('Default backend:', jax.default_backend())
print('CPU cores     :', os.cpu_count())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
JAX version   : 0.7.2
Devices       : [CpuDevice(id=0)]
Default backend: cpu
CPU cores     : 2


In [2]:
# ---------------------------------------------------------------------------
# Drive checkpoint helpers — atomic write via tmp-file + os.replace
# ---------------------------------------------------------------------------
DRIVE_DIR  = '/content/drive/MyDrive/ODE_bench_results'
CKPT_MAIN  = os.path.join(DRIVE_DIR, 'all_results.pkl')
CKPT_SCALE = os.path.join(DRIVE_DIR, 'scale_records.pkl')
CKPT_LD    = os.path.join(DRIVE_DIR, 'ld_records.pkl')

def _ensure_drive_dir():
    if IN_COLAB:
        os.makedirs(DRIVE_DIR, exist_ok=True)

def save_to_drive(obj, path):
    """Atomically pickle obj to path on Drive (no-op outside Colab)."""
    if not IN_COLAB:
        return
    _ensure_drive_dir()
    tmp = path + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    os.replace(tmp, path)
    print(f'  [ckpt] saved → {os.path.basename(path)}')

def load_from_drive(path):
    """Load pickle from Drive; returns None if not in Colab or file absent."""
    if not IN_COLAB or not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)

print('Checkpoint helpers defined')
print(f'Drive dir : {DRIVE_DIR}')


Checkpoint helpers defined
Drive dir : /content/drive/MyDrive/ODE_bench_results


## 1. Problem Interface

In [3]:
@dataclass
class ODEProblem:
    """Common interface for all ODE test problems."""
    name: str
    state_dim: int                              # d per trajectory
    t_span: Tuple[float, float]
    rhs: Callable                               # (t, y, params) -> dy/dt  [JAX-compatible]
    jac: Optional[Callable]                     # (t, y, params) -> d×d Jacobian
    ic_generator: Callable                      # (rng, n_traj) -> (n,d) array
    param_sampler: Callable                     # (rng, n_traj) -> params array
    ref_solution: Optional[Callable]            # (t_end, y0, params) -> y(t_end)
    # stiffness metadata
    stiffness_label: str = 'unknown'            # 'nonstiff'/'mild'/'strong'/'mixed'/'extreme'
    expected_stiffness_ratio: float = 1.0       # rough |λ_max/λ_min|
    # recommended solver hint
    solver_hint: str = 'explicit'               # 'explicit' | 'implicit'


def _check_rng(rng):
    if isinstance(rng, np.random.Generator):
        return rng
    return np.random.default_rng(rng)

## 2. Problem Library

In [4]:
# ---------------------------------------------------------------------------
# NS: Nonstiff harmonic oscillator  y'' + ω²y = 0
#     State: [y, y'], params: [ω] per mode, d modes decoupled
#     Exact: y_i(t) = a_i cos(ω_i t) + b_i sin(ω_i t)
# ---------------------------------------------------------------------------
def _ns_rhs(t, y, params):
    # y shape (2d,): [y0,y0',y1,y1',...]
    d = params.shape[0]
    yp = y.reshape(d, 2)
    dy = jnp.stack([
        yp[:, 1],
        -params**2 * yp[:, 0]
    ], axis=1).reshape(-1)
    return dy

def _ns_ref(t_end, y0, params):
    d = params.shape[0]
    y0r = y0.reshape(d, 2)
    a, b = y0r[:, 0], y0r[:, 1] / params
    ct, st = np.cos(params * t_end), np.sin(params * t_end)
    y_end = np.stack([a*ct + b*st, (-a*params*st + b*params*ct)], axis=1)
    return y_end.reshape(-1)

def make_nonstiff_problem(d=4, t_end=10.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.normal(size=(n, 2*d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(1.0, 4.0, size=(n, d)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        return np.array([_ns_ref(t_end, y0_batch[i], params_batch[i])
                         for i in range(len(y0_batch))])
    return ODEProblem(
        name='NS', state_dim=2*d, t_span=(0., t_end),
        rhs=_ns_rhs, jac=None,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='nonstiff', expected_stiffness_ratio=1.0,
        solver_hint='explicit'
    )

print('NS problem defined')

NS problem defined


In [5]:
# ---------------------------------------------------------------------------
# MS: Mildly stiff decoupled linear decay
#     y_i' = -λ_i (y_i - cos t) - sin t,  λ_i ∈ [1, 10]
# ---------------------------------------------------------------------------
def _stiff_lin_rhs(t, y, params):
    return -params * (y - jnp.cos(t)) - jnp.sin(t)

def _stiff_lin_jac(t, y, params):
    return -jnp.diag(params)

def _stiff_lin_ref(t_end, y0, params):
    return (y0 - 1.0) * np.exp(-params * t_end) + np.cos(t_end)

def make_mildly_stiff_problem(d=4, t_end=6.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.normal(size=(n, d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(1.0, 10.0, size=(n, d)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        return np.array([_stiff_lin_ref(t_end, y0_batch[i], params_batch[i])
                         for i in range(len(y0_batch))])
    return ODEProblem(
        name='MS', state_dim=d, t_span=(0., t_end),
        rhs=_stiff_lin_rhs, jac=_stiff_lin_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='mild', expected_stiffness_ratio=10.0,
        solver_hint='explicit'
    )


# ---------------------------------------------------------------------------
# SS: Strongly stiff decoupled linear decay
#     Same RHS, λ_i ∈ [100, 2000]
# ---------------------------------------------------------------------------
def make_strongly_stiff_problem(d=4, t_end=1.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.normal(size=(n, d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(100.0, 2000.0, size=(n, d)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        return np.array([_stiff_lin_ref(t_end, y0_batch[i], params_batch[i])
                         for i in range(len(y0_batch))])
    return ODEProblem(
        name='SS', state_dim=d, t_span=(0., t_end),
        rhs=_stiff_lin_rhs, jac=_stiff_lin_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='strong', expected_stiffness_ratio=2000.0,
        solver_hint='implicit'
    )

print('MS, SS problems defined')

MS, SS problems defined


In [6]:
# ---------------------------------------------------------------------------
# MX: Mixed-stiffness — fast modes (λ_fast) coupled to slow modes (λ_slow)
#     y_fast_i' = -λ_fast_i * y_fast_i + ε * y_slow_i
#     y_slow_i' = -λ_slow_i * y_slow_i
#     params: [λ_fast (d,), λ_slow (d,)] concatenated → (2d,)
# ---------------------------------------------------------------------------
def _mx_rhs(t, y, params):
    d = params.shape[0] // 2
    lf, ls = params[:d], params[d:]
    yf, ys = y[:d], y[d:]
    eps = 0.01
    dyf = -lf * yf + eps * ys
    dys = -ls * ys
    return jnp.concatenate([dyf, dys])

def _mx_jac(t, y, params):
    d = params.shape[0] // 2
    lf, ls = params[:d], params[d:]
    eps = 0.01
    J = jnp.zeros((2*d, 2*d))
    J = J.at[:d, :d].set(-jnp.diag(lf))
    J = J.at[:d, d:].set(eps * jnp.eye(d))
    J = J.at[d:, d:].set(-jnp.diag(ls))
    return J

def make_mixed_stiffness_problem(d=4, t_end=2.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(0.5, 2.0, size=(n, 2*d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        lf = rng.uniform(500.0, 1000.0, size=(n, d))
        ls = rng.uniform(0.5,   5.0,    size=(n, d))
        return np.concatenate([lf, ls], axis=1).astype(np.float64)
    return ODEProblem(
        name='MX', state_dim=2*d, t_span=(0., t_end),
        rhs=_mx_rhs, jac=_mx_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=None,
        stiffness_label='mixed', expected_stiffness_ratio=1000.0,
        solver_hint='implicit'
    )

print('MX problem defined')

MX problem defined


In [7]:
# ---------------------------------------------------------------------------
# VZ: Van der Pol oscillator (stiffness increases with μ)
#     y1' = y2
#     y2' = μ(1 - y1²)y2 - y1
#     params: [μ],  μ ∈ [1, 50]
# ---------------------------------------------------------------------------
def _vz_rhs(t, y, params):
    mu = params[0]
    return jnp.array([y[1], mu * (1. - y[0]**2) * y[1] - y[0]])

def _vz_jac(t, y, params):
    mu = params[0]
    return jnp.array([
        [0.,                          1.],
        [-2.*mu*y[0]*y[1] - 1.,  mu*(1. - y[0]**2)]
    ])

def make_vanderpol_problem(t_end=10.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        y1 = rng.uniform(1.5, 2.5, size=(n, 1))
        y2 = rng.uniform(-1., 1., size=(n, 1))
        return np.concatenate([y1, y2], axis=1).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(1.0, 50.0, size=(n, 1)).astype(np.float64)
    return ODEProblem(
        name='VZ', state_dim=2, t_span=(0., t_end),
        rhs=_vz_rhs, jac=_vz_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=None,
        stiffness_label='strong', expected_stiffness_ratio=50.0,
        solver_hint='implicit'
    )

print('VZ problem defined')

VZ problem defined


In [8]:
# ---------------------------------------------------------------------------
# RB: Robertson chemical kinetics (extremely stiff, 3 species)
#     y1' = -k1*y1 + k3*y2*y3
#     y2' =  k1*y1 - k2*y2^2 - k3*y2*y3
#     y3' =  k2*y2^2
#     params: [k1, k2, k3]  (nominal: 0.04, 3e7, 1e4; varied ±25%)
#     t_end=100 — avoids h_min stall; at 1e4 the trapezoidal error estimate
#     forces h→h_min near t=0 (BE cannot span the full 12-decade λ range).
# ---------------------------------------------------------------------------
def _rb_rhs(t, y, params):
    k1, k2, k3 = params[0], params[1], params[2]
    y1, y2, y3 = y[0], y[1], y[2]
    return jnp.array([
        -k1*y1 + k3*y2*y3,
         k1*y1 - k2*y2**2 - k3*y2*y3,
         k2*y2**2
    ])

def _rb_jac(t, y, params):
    k1, k2, k3 = params[0], params[1], params[2]
    y1, y2, y3 = y[0], y[1], y[2]
    return jnp.array([
        [-k1,           k3*y3,         k3*y2],
        [ k1, -2.*k2*y2-k3*y3,        -k3*y2],
        [0.,        2.*k2*y2,           0.  ]
    ])

def make_robertson_problem(t_end=100.0):
    def ic_gen(rng, n):
        return np.tile([1., 0., 0.], (n, 1)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        scale   = rng.uniform(0.75, 1.25, size=(n, 3))
        nominal = np.array([0.04, 3e7, 1e4])
        return (scale * nominal).astype(np.float64)
    return ODEProblem(
        name='RB', state_dim=3, t_span=(0., t_end),
        rhs=_rb_rhs, jac=_rb_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=None,
        stiffness_label='extreme', expected_stiffness_ratio=1e10,
        solver_hint='implicit'
    )

print('RB problem defined')


RB problem defined


In [9]:
# ---------------------------------------------------------------------------
# LD: Large-dimensional stiff linear decay  (d=64)
#     Tests GPU register pressure and batched matrix-solve scaling.
#     RHS/Jac reuse _stiff_lin_rhs / _stiff_lin_jac from the MS/SS cell.
# ---------------------------------------------------------------------------
def make_large_dim_problem(d=64, t_end=1.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.normal(size=(n, d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(5.0, 500.0, size=(n, d)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        return np.array([_stiff_lin_ref(t_end, y0_batch[i], params_batch[i])
                         for i in range(len(y0_batch))])
    return ODEProblem(
        name='LD', state_dim=d, t_span=(0., t_end),
        rhs=_stiff_lin_rhs, jac=_stiff_lin_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='strong', expected_stiffness_ratio=100.0,
        solver_hint='implicit'
    )

print('LD problem defined')


LD problem defined


In [10]:
# ---------------------------------------------------------------------------
# LZ: Lorenz attractor — chaotic, mildly stiff, extreme step-size divergence
#     x' = σ(y - x),  y' = x(ρ-z) - y,  z' = xy - βz
#     params: [σ, ρ, β]   σ=10 fixed, ρ ∈ [20,35], β=8/3 fixed
#     Batch over ρ and IC noise: chaotic sensitivity → rapid step-size spread
#     across the batch.  Good stress test for COMPACTION and BUCKETS.
# ---------------------------------------------------------------------------
def _lz_rhs(t, u, params):
    sig, rho, beta = params[0], params[1], params[2]
    return jnp.array([
        sig  * (u[1] - u[0]),
        u[0] * (rho - u[2]) - u[1],
        u[0] * u[1] - beta * u[2],
    ])

def _lz_jac(t, u, params):
    sig, rho, beta = params[0], params[1], params[2]
    return jnp.array([
        [-sig,          sig,       0.   ],
        [rho - u[2],   -1.,       -u[0] ],
        [u[1],          u[0],     -beta ],
    ])

def make_lorenz_problem(t_end=10.0):
    def ic_gen(rng, n):
        rng  = _check_rng(rng)
        base = np.array([1.0, 0.0, 0.0])
        return (base + rng.normal(scale=0.5, size=(n, 3))).astype(np.float64)
    def param_sampler(rng, n):
        rng  = _check_rng(rng)
        sig  = np.full(n, 10.0)
        rho  = rng.uniform(20.0, 35.0, size=n)
        beta = np.full(n, 8.0 / 3.0)
        return np.stack([sig, rho, beta], axis=1).astype(np.float64)
    return ODEProblem(
        name='LZ', state_dim=3, t_span=(0., t_end),
        rhs=_lz_rhs, jac=_lz_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=None,
        stiffness_label='mild', expected_stiffness_ratio=35.0,
        solver_hint='explicit',
    )

print('LZ (Lorenz attractor) problem defined')


LZ (Lorenz attractor) problem defined


In [11]:
# ---------------------------------------------------------------------------
# HE: 1D Heat equation — method-of-lines  (parabolic PDE → stiff ODE)
#     u_t = α u_xx,  x ∈ (0,1),  u(0,t) = u(1,t) = 0
#     Spatial:  N=20 interior points,  Δx = 1/(N+1)
#     Scheme:   second-order finite difference → tridiagonal Jacobian
#     IC:       u(x,0) = A·sin(πx),  A ∈ [0.5, 2.0]
#     Exact ODE solution (single-mode IC = A·sin(πxᵢ)):
#       yᵢ(t) = yᵢ(0) · exp(λ₁ t)
#       λ₁ = −2(α/Δx²)·(1−cos(πΔx))   ← exact k=1 eigenvalue of tridiag matrix
#     Params:  [α],  α ∈ [0.1, 2.0]
#     Stiffness ratio ≈ 4(N+1)²/π² × α ≈ 178α   (strong for large α)
#     Key GPU test: batched 20×20 dense linear solve per Newton step.
# ---------------------------------------------------------------------------
_HE_N = 20
_HE_h = 1.0 / (_HE_N + 1)    # Δx = 1/21

def _he_rhs(t, y, params):
    alpha   = params[0]
    c       = alpha / (_HE_h * _HE_h)
    y_left  = jnp.concatenate([jnp.zeros(1), y[:-1]])   # ghost 0 at left BC
    y_right = jnp.concatenate([y[1:], jnp.zeros(1)])    # ghost 0 at right BC
    return c * (y_left - 2.0 * y + y_right)

def _he_jac(t, y, params):
    alpha = params[0]
    c     = alpha / (_HE_h * _HE_h)
    diag  = jnp.full(_HE_N, -2.0 * c)
    off   = jnp.full(_HE_N - 1, c)
    return jnp.diag(diag) + jnp.diag(off, 1) + jnp.diag(off, -1)

def make_heat_eq_problem(t_end=0.5):
    x_np = np.arange(1, _HE_N + 1) * _HE_h
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        A   = rng.uniform(0.5, 2.0, size=(n, 1))
        return (A * np.sin(np.pi * x_np)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(0.1, 2.0, size=(n, 1)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        # Exact discrete (ODE) solution: yᵢ(t) = yᵢ(0) · exp(λ₁·t)
        alpha  = params_batch[:, 0]
        c      = alpha / (_HE_h * _HE_h)
        lam1   = -2.0 * c * (1.0 - np.cos(np.pi * _HE_h))   # exact eigenvalue
        decay  = np.exp(lam1 * t_end)
        return y0_batch * decay[:, None]
    return ODEProblem(
        name='HE', state_dim=_HE_N, t_span=(0., t_end),
        rhs=_he_rhs, jac=_he_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='strong',
        expected_stiffness_ratio=178.0,
        solver_hint='implicit',
    )

print('HE (1D Heat Eq MoL, d=20, tridiag Jac) problem defined')


HE (1D Heat Eq MoL, d=20, tridiag Jac) problem defined


In [12]:
# ---------------------------------------------------------------------------
# AD: 1D Advection equation — method-of-lines (hyperbolic PDE → nonstiff ODE)
#     u_t + c·u_x = 0,  x ∈ [0,1),  periodic BC
#     Spatial:  N=20 grid points, Δx = 1/N, first-order upwind
#     Scheme:   duᵢ/dt = −(c/Δx)·(uᵢ − uᵢ₋₁),   u₋₁ ≡ u_{N-1}
#     IC:       u(x,0) = A·sin(2πx),  A ∈ [0.5, 1.5]
#     Exact ODE solution via DFT of discrete eigenvalues:
#       λₖ = −(c/Δx)·(1−e^{−2πik/N}),   y(t) = IDFT(e^{λt}·DFT(y₀))
#     Params:  [c],  c ∈ [0.2, 2.0]  (positive wave speed → upwind stable)
#     Step-size spread: h_stable ∝ Δx/c → 10× spread for 10× range in c.
#     Key GPU test: explicit solver under maximum step-size divergence.
# ---------------------------------------------------------------------------
_AD_N  = 20
_AD_dx = 1.0 / _AD_N

def _ad_rhs(t, y, params):
    c      = params[0]
    y_left = jnp.roll(y, 1)           # periodic: y_{-1} = y_{N-1}
    return -(c / _AD_dx) * (y - y_left)

def _ad_jac(t, y, params):
    # J[i,i] = -coh,  J[i, (i-1)%N] = +coh  (upwind, periodic BC)
    # roll(I, -1, axis=1)[i,j] = 1  iff  j = (i-1) mod N
    c   = params[0]
    coh = c / _AD_dx
    return -coh * jnp.eye(_AD_N) + coh * jnp.roll(jnp.eye(_AD_N), -1, axis=1)

def make_advection_problem(t_end=1.0):
    x_np = np.arange(_AD_N) / _AD_N
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        A   = rng.uniform(0.5, 1.5, size=(n, 1))
        return (A * np.sin(2.0 * np.pi * x_np)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(0.2, 2.0, size=(n, 1)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        # Exact discrete-system solution via DFT propagation
        k_idx   = np.arange(_AD_N)
        results = np.empty_like(y0_batch)
        for i in range(len(y0_batch)):
            c_i  = params_batch[i, 0]
            lam  = -(c_i / _AD_dx) * (1.0 - np.exp(-2j * np.pi * k_idx / _AD_N))
            y0_k = np.fft.fft(y0_batch[i])
            results[i] = np.real(np.fft.ifft(y0_k * np.exp(lam * t_end)))
        return results
    return ODEProblem(
        name='AD', state_dim=_AD_N, t_span=(0., t_end),
        rhs=_ad_rhs, jac=_ad_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='nonstiff',
        expected_stiffness_ratio=10.0,
        solver_hint='explicit',
    )

print('AD (1D Advection MoL, d=20, periodic BC) problem defined')


AD (1D Advection MoL, d=20, periodic BC) problem defined


In [13]:
# ---------------------------------------------------------------------------
# Problem registry — all 10 problems
# ---------------------------------------------------------------------------
PROBLEMS = {
    'NS': make_nonstiff_problem(d=4,     t_end=10.0),   # nonstiff harmonic oscillator
    'MS': make_mildly_stiff_problem(d=4, t_end=6.0),    # mild stiffness  λ∈[1,10]
    'SS': make_strongly_stiff_problem(d=4, t_end=1.0),  # strong stiffness λ∈[100,2000]
    'MX': make_mixed_stiffness_problem(d=4, t_end=2.0), # fast+slow coupled modes
    'VZ': make_vanderpol_problem(t_end=10.0),            # Van der Pol μ∈[1,50]
    'RB': make_robertson_problem(t_end=100.0),           # Robertson kinetics (extreme)
    'LD': make_large_dim_problem(d=64,   t_end=1.0),    # d=64 batched linalg stress
    'LZ': make_lorenz_problem(t_end=10.0),               # Lorenz chaos, h-spread test
    'HE': make_heat_eq_problem(t_end=0.5),               # PDE parabolic: 1D heat MoL
    'AD': make_advection_problem(t_end=1.0),             # PDE hyperbolic: 1D advection MoL
}

print(f'{"ID":>4} {"dim":>5} {"stiffness":>10} {"hint":>10} {"t_end":>8}  has_ref')
print('-' * 52)
for k, p in PROBLEMS.items():
    has_ref = p.ref_solution is not None
    print(f'{k:>4} {p.state_dim:>5} {p.stiffness_label:>10} {p.solver_hint:>10} '
          f'{p.t_span[1]:>8.3g}  {"yes" if has_ref else "no"}')


  ID   dim  stiffness       hint    t_end  has_ref
----------------------------------------------------
  NS     8   nonstiff   explicit       10  yes
  MS     4       mild   explicit        6  yes
  SS     4     strong   implicit        1  yes
  MX     8      mixed   implicit        2  no
  VZ     2     strong   implicit       10  no
  RB     3    extreme   implicit      100  no
  LD    64     strong   implicit        1  yes
  LZ     3       mild   explicit       10  no
  HE    20     strong   implicit      0.5  yes
  AD    20   nonstiff   explicit        1  yes


## 3. Core JAX Solvers (problem-agnostic)

In [14]:
# ---------------------------------------------------------------------------
# Instrumentation dataclass -- accumulated during a solver run
# ---------------------------------------------------------------------------
from dataclasses import dataclass as _dc, field as _f

@_dc
class SolverStats:
    n_steps_total: int = 0
    active_fracs: list  = _f(default_factory=list)   # per-step active fraction
    h_covs: list        = _f(default_factory=list)   # per-step CoV of h across active
    warp_h_covs: list   = _f(default_factory=list)   # per-cluster mean within-warp h-CoV
    gpu_util_samples: list   = _f(default_factory=list)  # [(t_s, util_pct), ...]
    gpu_mem_mb_samples: list = _f(default_factory=list)  # [(t_s, mem_mb), ...]
    n_accept: np.ndarray = None
    n_reject: np.ndarray = None
    n_newton: np.ndarray = None
    wall_time: float = 0.0
    compaction_overhead: float = 0.0
    batch_size: int = 0

    def summary(self):
        af  = np.array(self.active_fracs)
        hc  = np.array(self.h_covs)
        rej = (self.n_reject.sum() /
               max(self.n_accept.sum() + self.n_reject.sum(), 1))
        gpu_utils = [u for _, u in self.gpu_util_samples]
        gpu_mems  = [m for _, m in self.gpu_mem_mb_samples]
        return {
            "wall_time":           self.wall_time,
            "n_steps_total":       self.n_steps_total,
            "mean_active_frac":    float(af.mean())  if len(af) else 0.,
            "min_active_frac":     float(af.min())   if len(af) else 0.,
            "mean_h_cov":          float(hc.mean())  if len(hc) else 0.,
            "max_h_cov":           float(hc.max())   if len(hc) else 0.,
            "rej_rate":            float(rej),
            "mean_accept":         float(self.n_accept.mean()),
            "mean_newton":         float(self.n_newton.mean()) if self.n_newton is not None else 0.,
            "compaction_overhead": self.compaction_overhead,
            "mean_warp_h_cov":     float(np.mean(self.warp_h_covs)) if self.warp_h_covs else 0.,
            "mean_gpu_util":       float(np.mean(gpu_utils))  if gpu_utils else 0.,
            "min_gpu_util":        float(np.min(gpu_utils))   if gpu_utils else 0.,
            "peak_gpu_mem_mb":     float(np.max(gpu_mems))    if gpu_mems  else 0.,
            "throughput":          float(self.batch_size / max(self.wall_time, 1e-9)),
        }

    def print_summary(self, label=""):
        s = self.summary()
        print(f"{label}")
        print(f'  wall={s["wall_time"]:.3f}s  steps={s["n_steps_total"]}  '
              f'active={s["mean_active_frac"]:.2%}(min {s["min_active_frac"]:.2%})')
        print(f'  h_cov={s["mean_h_cov"]:.3f}(max {s["max_h_cov"]:.3f})  '
              f'rej_rate={s["rej_rate"]:.2%}  '
              f'mean_newton={s["mean_newton"]:.2f}  '
              f'compact_oh={s["compaction_overhead"]:.3f}s  '
              f'warp_h_cov={s["mean_warp_h_cov"]:.4f}')
        print(f'  throughput={s["throughput"]:.1f} traj/s  '
              f'gpu_util={s["mean_gpu_util"]:.1f}%(min {s["min_gpu_util"]:.1f}%)  '
              f'peak_mem={s["peak_gpu_mem_mb"]:.0f}MB')

print("SolverStats defined")


SolverStats defined


In [15]:
# ---------------------------------------------------------------------------
# Explicit RK45 (Dormand-Prince) — problem-agnostic via rhs callable
# ---------------------------------------------------------------------------
def _rk45_step(rhs, t, y, h, params):
    """Single DP5(4) step for one trajectory."""
    k1 = rhs(t,          y,                                                  params)
    k2 = rhs(t+h/5,      y+h*(1/5*k1),                                       params)
    k3 = rhs(t+3*h/10,   y+h*(3/40*k1+9/40*k2),                              params)
    k4 = rhs(t+4*h/5,    y+h*(44/45*k1-56/15*k2+32/9*k3),                    params)
    k5 = rhs(t+8*h/9,    y+h*(19372/6561*k1-25360/2187*k2
                               +64448/6561*k3-212/729*k4),                    params)
    k6 = rhs(t+h,        y+h*(9017/3168*k1-355/33*k2+46732/5247*k3
                               +49/176*k4-5103/18656*k5),                     params)
    y5 = y+h*(35/384*k1+500/1113*k3+125/192*k4-2187/6784*k5+11/84*k6)
    k7 = rhs(t+h, y5, params)
    y4 = y+h*(5179/57600*k1+7571/16695*k3+393/640*k4
              -92097/339200*k5+187/2100*k6+1/40*k7)
    return y5, y5-y4

def _err_norm(y_old, y_new, err, rtol, atol):
    sc = atol + rtol * jnp.maximum(jnp.abs(y_old), jnp.abs(y_new))
    return jnp.sqrt(jnp.mean((err/sc)**2, axis=-1))

print('RK45 step function defined')

RK45 step function defined


In [16]:
# ---------------------------------------------------------------------------
# Implicit BE (BDF1) Newton solver — problem-agnostic
# ---------------------------------------------------------------------------
def _newton_be(rhs, jac, y_prev, t_next, h, params, tol=1e-10, max_iter=20):
    """Batched backward-Euler Newton. Returns (y, converged, n_iters)."""
    B, d  = y_prev.shape
    eye   = jnp.eye(d)
    y     = y_prev.copy()
    conv  = jnp.zeros((B,), dtype=bool)
    iters = jnp.zeros((B,), jnp.int32)
    for k in range(max_iter):
        f   = jax.vmap(rhs, in_axes=(0, 0, 0))(t_next, y, params)
        res = y - y_prev - h[:, None] * f
        rn  = jnp.max(jnp.abs(res), axis=1)
        jc  = (~conv) & (rn < tol)
        iters = jnp.where(jc, k+1, iters)
        conv  = conv | jc
        if bool(jnp.all(conv)): break
        Jf  = jax.vmap(jac, in_axes=(0, 0, 0))(t_next, y, params)
        A   = eye[None] - h[:, None, None] * Jf
        dy  = jnp.linalg.solve(A, -res[..., None]).squeeze(-1)
        dy  = jnp.where(conv[:, None], 0., dy)
        y   = y + dy
        dn  = jnp.max(jnp.abs(dy), axis=1)
        jc2 = (~conv) & (dn < tol)
        iters = jnp.where(jc2, k+2, iters)
        conv  = conv | jc2
    iters = jnp.where((iters == 0) & conv, 1, iters)
    iters = jnp.where(~conv, max_iter, iters)
    return y, conv, iters

print('Newton BE solver defined')

Newton BE solver defined


## 4. GPU Balancing Strategies

In [17]:
# ---------------------------------------------------------------------------
# Strategy 1: BASELINE — masked batch (existing approach, no reordering)
# ---------------------------------------------------------------------------
def solve_baseline_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    t0, tf = problem.t_span
    rhs = problem.rhs
    use_implicit = (problem.solver_hint == 'implicit') and (problem.jac is not None)

    y      = jnp.array(y0_batch, dtype=jnp.float64)
    params = jnp.array(params_batch, dtype=jnp.float64)
    B      = y.shape[0]
    t      = jnp.full((B,), t0)
    h      = jnp.full((B,), h_init)
    done   = jnp.zeros((B,), dtype=bool)
    n_acc  = jnp.zeros((B,), jnp.int32)
    n_rej  = jnp.zeros((B,), jnp.int32)
    n_nit  = jnp.zeros((B,), jnp.int32)

    stats = SolverStats()
    t_start = time.perf_counter()

    for step in range(max_iters):
        active = ~done
        n_active = int(jnp.sum(active))
        if n_active == 0:
            break

        # Instrumentation
        stats.active_fracs.append(n_active / B)
        h_active = np.array(h)[np.array(active)]
        if len(h_active) > 1:
            stats.h_covs.append(float(h_active.std() / max(h_active.mean(), 1e-300)))
        else:
            stats.h_covs.append(0.0)

        h_eff = jnp.clip(
            jnp.where(active, jnp.minimum(h, tf - t), h),
            h_min, h_max
        )

        if use_implicit:
            tn = t + h_eff
            yn, ok, ni = _newton_be(rhs, problem.jac, y, tn, h_eff, params,
                                    tol=newton_tol, max_iter=newton_max_iter)
            n_nit = n_nit + ni
            # simple error estimate: norm of Newton residual proxy
            f_old = jax.vmap(rhs, in_axes=(0, 0, 0))(t, y, params)
            f_new = jax.vmap(rhs, in_axes=(0, 0, 0))(tn, yn, params)
            err_est = _err_norm(y, yn, h_eff[:, None] * (f_new - f_old) / 2., rtol, atol)
            acc = ok & active & (err_est <= 1.)
            t   = jnp.where(acc, tn, t)
            y   = jnp.where(acc[:, None], yn, y)
            se  = jnp.where(err_est > 0, err_est, 1e-300)
            fac = jnp.clip(safety * se**(-0.5), min_factor, max_factor)
            fac_rej = jnp.clip(safety * se**(-0.5), min_factor, 1.0)
            h   = jnp.where(active,
                            jnp.clip(jnp.where(acc, h_eff*fac, h_eff*fac_rej), h_min, h_max),
                            h)
            n_acc = n_acc + acc.astype(jnp.int32)
            n_rej = n_rej + (~acc & active).astype(jnp.int32)
        else:
            step_fn = lambda t_, y_, h_, p_: _rk45_step(rhs, t_, y_, h_, p_)
            yc, ev  = jax.vmap(step_fn)(t, y, h_eff, params)
            err     = _err_norm(y, yc, ev, rtol, atol)
            acc     = (err <= 1.) & active
            t       = jnp.where(acc, t + h_eff, t)
            y       = jnp.where(acc[:, None], yc, y)
            se      = jnp.where(err > 0, err, 1e-300)
            fa      = jnp.where(err == 0., max_factor,
                                jnp.clip(safety*se**(-0.2), min_factor, max_factor))
            fr      = jnp.clip(safety*se**(-0.2), min_factor, 1.)
            h       = jnp.where(active,
                                jnp.clip(jnp.where(acc, h_eff*fa, h_eff*fr), h_min, h_max),
                                h)
            n_acc = n_acc + acc.astype(jnp.int32)
            n_rej = n_rej + (~acc & active).astype(jnp.int32)

        done = done | (t >= tf - 1e-12)
        stats.n_steps_total += 1

    stats.wall_time  = time.perf_counter() - t_start
    stats.n_accept   = np.array(n_acc)
    stats.n_reject   = np.array(n_rej)
    stats.n_newton   = np.array(n_nit)
    return np.array(y), stats

print('Baseline solver defined')

Baseline solver defined


In [18]:
# ---------------------------------------------------------------------------
# Strategy 2: COMPACTION — repack only active trajectories each iteration
# ---------------------------------------------------------------------------
def solve_compaction_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    t0, tf = problem.t_span
    rhs = problem.rhs
    use_implicit = (problem.solver_hint == 'implicit') and (problem.jac is not None)

    B          = len(y0_batch)
    y_full     = np.array(y0_batch, dtype=np.float64)
    p_full     = np.array(params_batch, dtype=np.float64)
    t_full     = np.full(B, float(t0))
    h_full     = np.full(B, h_init)
    done_full  = np.zeros(B, dtype=bool)
    n_acc_full = np.zeros(B, dtype=np.int32)
    n_rej_full = np.zeros(B, dtype=np.int32)
    n_nit_full = np.zeros(B, dtype=np.int32)
    # index mapping: active_indices[i] = original trajectory index
    active_idx = np.arange(B)

    stats = SolverStats()
    t_start = time.perf_counter()
    t_compact = 0.0

    # Working arrays (compacted)
    y  = jnp.array(y_full)
    p  = jnp.array(p_full)
    t  = jnp.array(t_full)
    h  = jnp.array(h_full)
    n_acc = jnp.zeros(B, jnp.int32)
    n_rej = jnp.zeros(B, jnp.int32)
    n_nit = jnp.zeros(B, jnp.int32)

    for step in range(max_iters):
        Bc = int(y.shape[0])
        if Bc == 0:
            break

        stats.active_fracs.append(Bc / B)
        h_np = np.array(h)
        stats.h_covs.append(float(h_np.std() / max(h_np.mean(), 1e-300)) if Bc > 1 else 0.)

        h_eff = jnp.clip(jnp.minimum(h, tf - t), h_min, h_max)

        if use_implicit:
            tn  = t + h_eff
            yn, ok, ni = _newton_be(rhs, problem.jac, y, tn, h_eff, p,
                                    tol=newton_tol, max_iter=newton_max_iter)
            n_nit = n_nit + ni
            f_new = jax.vmap(rhs, in_axes=(0, 0, 0))(tn, yn, p)
            f_old = jax.vmap(rhs, in_axes=(0, 0, 0))(t,  y,  p)
            ee    = _err_norm(y, yn, h_eff[:, None]*(f_new-f_old)/2., rtol, atol)
            acc   = ok & (ee <= 1.)
            t     = jnp.where(acc, tn, t)
            y     = jnp.where(acc[:, None], yn, y)
            se    = jnp.where(ee > 0, ee, 1e-300)
            fac     = jnp.clip(safety*se**(-0.5), min_factor, max_factor)
            fac_rej = jnp.clip(safety*se**(-0.5), min_factor, 1.0)
            h     = jnp.clip(jnp.where(acc, h_eff*fac, h_eff*fac_rej), h_min, h_max)
        else:
            yc, ev = jax.vmap(lambda t_,y_,h_,p_: _rk45_step(rhs,t_,y_,h_,p_))(t,y,h_eff,p)
            err    = _err_norm(y, yc, ev, rtol, atol)
            acc    = err <= 1.
            t      = jnp.where(acc, t + h_eff, t)
            y      = jnp.where(acc[:, None], yc, y)
            se     = jnp.where(err > 0, err, 1e-300)
            fa     = jnp.where(err == 0., max_factor,
                               jnp.clip(safety*se**(-0.2), min_factor, max_factor))
            fr     = jnp.clip(safety*se**(-0.2), min_factor, 1.)
            h      = jnp.clip(jnp.where(acc, h_eff*fa, h_eff*fr), h_min, h_max)

        n_acc = n_acc + acc.astype(jnp.int32)
        n_rej = n_rej + (~acc).astype(jnp.int32)
        stats.n_steps_total += 1

        # Compaction: remove finished trajectories
        tc0 = time.perf_counter()
        t_np  = np.array(t)
        finished = t_np >= tf - 1e-12
        if np.any(finished):
            fin_orig = active_idx[finished]
            y_full[fin_orig]     = np.array(y)[finished]
            t_full[fin_orig]     = t_np[finished]
            n_acc_full[fin_orig] = np.array(n_acc)[finished]
            n_rej_full[fin_orig] = np.array(n_rej)[finished]
            n_nit_full[fin_orig] = np.array(n_nit)[finished]
            keep = ~finished
            active_idx = active_idx[keep]
            y     = jnp.array(np.array(y)[keep])
            p     = jnp.array(np.array(p)[keep])
            t     = jnp.array(t_np[keep])
            h     = jnp.array(np.array(h)[keep])
            n_acc = jnp.array(np.array(n_acc)[keep])
            n_rej = jnp.array(np.array(n_rej)[keep])
            n_nit = jnp.array(np.array(n_nit)[keep])
        t_compact += time.perf_counter() - tc0

    # Flush remaining
    if len(active_idx) > 0:
        y_full[active_idx]     = np.array(y)
        n_acc_full[active_idx] = np.array(n_acc)
        n_rej_full[active_idx] = np.array(n_rej)
        n_nit_full[active_idx] = np.array(n_nit)

    stats.wall_time           = time.perf_counter() - t_start
    stats.compaction_overhead = t_compact
    stats.n_accept            = n_acc_full
    stats.n_reject            = n_rej_full
    stats.n_newton            = n_nit_full
    return y_full, stats

print('Compaction solver defined')

Compaction solver defined


In [19]:
# ---------------------------------------------------------------------------
# Strategy 3: REBATCH — compact every K steps instead of every step
# ---------------------------------------------------------------------------
def solve_rebatch_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    rebatch_every: int = 50,
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    t0, tf = problem.t_span
    rhs = problem.rhs
    use_implicit = (problem.solver_hint == 'implicit') and (problem.jac is not None)

    B           = len(y0_batch)
    y_full      = np.array(y0_batch, dtype=np.float64)
    p_full      = np.array(params_batch, dtype=np.float64)
    done_full   = np.zeros(B, dtype=bool)
    n_acc_full  = np.zeros(B, dtype=np.int32)
    n_rej_full  = np.zeros(B, dtype=np.int32)
    n_nit_full  = np.zeros(B, dtype=np.int32)
    active_idx  = np.arange(B)

    y     = jnp.array(y_full)
    p     = jnp.array(p_full)
    t     = jnp.full(B, float(t0))
    h     = jnp.full(B, h_init)
    done  = jnp.zeros(B, dtype=bool)
    n_acc = jnp.zeros(B, jnp.int32)
    n_rej = jnp.zeros(B, jnp.int32)
    n_nit = jnp.zeros(B, jnp.int32)

    stats   = SolverStats()
    t_start = time.perf_counter()
    t_compact = 0.0

    for step in range(max_iters):
        active = ~done
        n_active = int(jnp.sum(active))
        if n_active == 0:
            break

        stats.active_fracs.append(n_active / B)
        h_act = np.array(h)[np.array(active)]
        stats.h_covs.append(float(h_act.std()/max(h_act.mean(),1e-300)) if len(h_act)>1 else 0.)

        h_eff = jnp.clip(
            jnp.where(active, jnp.minimum(h, tf - t), h),
            h_min, h_max
        )

        if use_implicit:
            tn  = t + h_eff
            yn, ok, ni = _newton_be(rhs, problem.jac, y, tn, h_eff, p,
                                    tol=newton_tol, max_iter=newton_max_iter)
            n_nit = n_nit + ni
            f_new = jax.vmap(rhs, in_axes=(0, 0, 0))(tn, yn, p)
            f_old = jax.vmap(rhs, in_axes=(0, 0, 0))(t,  y,  p)
            ee    = _err_norm(y, yn, h_eff[:, None]*(f_new-f_old)/2., rtol, atol)
            acc   = ok & active & (ee <= 1.)
            t     = jnp.where(acc, tn, t)
            y     = jnp.where(acc[:, None], yn, y)
            se    = jnp.where(ee > 0, ee, 1e-300)
            fac     = jnp.clip(safety*se**(-0.5), min_factor, max_factor)
            fac_rej = jnp.clip(safety*se**(-0.5), min_factor, 1.0)
            h     = jnp.where(active, jnp.clip(jnp.where(acc, h_eff*fac, h_eff*fac_rej), h_min, h_max), h)
        else:
            yc, ev = jax.vmap(lambda t_,y_,h_,p_: _rk45_step(rhs,t_,y_,h_,p_))(t,y,h_eff,p)
            err = _err_norm(y, yc, ev, rtol, atol)
            acc = (err <= 1.) & active
            t   = jnp.where(acc, t + h_eff, t)
            y   = jnp.where(acc[:, None], yc, y)
            se  = jnp.where(err > 0, err, 1e-300)
            fa  = jnp.where(err == 0., max_factor,
                            jnp.clip(safety*se**(-0.2), min_factor, max_factor))
            fr  = jnp.clip(safety*se**(-0.2), min_factor, 1.)
            h   = jnp.where(active, jnp.clip(jnp.where(acc, h_eff*fa, h_eff*fr), h_min, h_max), h)

        n_acc = n_acc + acc.astype(jnp.int32)
        n_rej = n_rej + (~acc & active).astype(jnp.int32)
        done  = done | (t >= tf - 1e-12)
        stats.n_steps_total += 1

        # Periodic compaction
        if (step + 1) % rebatch_every == 0 and bool(jnp.any(done)):
            tc0 = time.perf_counter()
            done_np = np.array(done)
            fin_orig = active_idx[done_np]
            y_full[fin_orig]     = np.array(y)[done_np]
            n_acc_full[fin_orig] = np.array(n_acc)[done_np]
            n_rej_full[fin_orig] = np.array(n_rej)[done_np]
            n_nit_full[fin_orig] = np.array(n_nit)[done_np]
            keep = ~done_np
            active_idx = active_idx[keep]
            y     = jnp.array(np.array(y)[keep])
            p     = jnp.array(np.array(p)[keep])
            t     = jnp.array(np.array(t)[keep])
            h     = jnp.array(np.array(h)[keep])
            n_acc = jnp.array(np.array(n_acc)[keep])
            n_rej = jnp.array(np.array(n_rej)[keep])
            n_nit = jnp.array(np.array(n_nit)[keep])
            done  = jnp.zeros(len(active_idx), dtype=bool)
            t_compact += time.perf_counter() - tc0

    if len(active_idx) > 0:
        y_full[active_idx]     = np.array(y)
        n_acc_full[active_idx] = np.array(n_acc)
        n_rej_full[active_idx] = np.array(n_rej)
        n_nit_full[active_idx] = np.array(n_nit)

    stats.wall_time           = time.perf_counter() - t_start
    stats.compaction_overhead = t_compact
    stats.n_accept            = n_acc_full
    stats.n_reject            = n_rej_full
    stats.n_newton            = n_nit_full
    return y_full, stats

print('Rebatch solver defined')

Rebatch solver defined


In [20]:
# ---------------------------------------------------------------------------
# Strategy 4: BUCKETS — partition batch by stiffness regime at t=0,
# then solve each sub-batch independently (avoids step-size spread divergence).
# Buckets determined by spread in estimated stiffness (max |λ| via param proxy).
# ---------------------------------------------------------------------------
def solve_buckets_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    n_buckets: int = 4,
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    B = len(y0_batch)
    t0, tf = problem.t_span

    # Stiffness proxy: max param value per trajectory
    stiff_proxy = np.max(np.abs(params_batch), axis=1)  # (B,)
    bucket_edges = np.percentile(stiff_proxy, np.linspace(0, 100, n_buckets + 1))
    bucket_ids   = np.digitize(stiff_proxy, bucket_edges[1:-1])

    y_full     = np.zeros_like(y0_batch)
    n_acc_full = np.zeros(B, dtype=np.int32)
    n_rej_full = np.zeros(B, dtype=np.int32)
    n_nit_full = np.zeros(B, dtype=np.int32)

    combined_stats = SolverStats()
    t_start = time.perf_counter()

    for bid in range(n_buckets):
        idx = np.where(bucket_ids == bid)[0]
        if len(idx) == 0:
            continue
        yb  = y0_batch[idx]
        pb  = params_batch[idx]
        # Each bucket gets an h_init tuned to its stiffness range
        sp_mean = stiff_proxy[idx].mean()
        h0  = float(np.clip(0.1 / max(sp_mean, 1.), h_min, h_max))
        y_out, st = solve_baseline_jax(
            problem, yb, pb,
            rtol=rtol, atol=atol,
            h_init=h0, h_min=h_min, h_max=h_max,
            newton_tol=newton_tol, newton_max_iter=newton_max_iter,
            max_iters=max_iters, safety=safety,
            min_factor=min_factor, max_factor=max_factor,
        )
        y_full[idx]     = y_out
        n_acc_full[idx] = st.n_accept
        n_rej_full[idx] = st.n_reject
        n_nit_full[idx] = st.n_newton
        combined_stats.n_steps_total  += st.n_steps_total
        combined_stats.active_fracs   += st.active_fracs
        combined_stats.h_covs         += st.h_covs

    combined_stats.wall_time = time.perf_counter() - t_start
    combined_stats.n_accept  = n_acc_full
    combined_stats.n_reject  = n_rej_full
    combined_stats.n_newton  = n_nit_full
    return y_full, combined_stats

print('Bucket solver defined')

Bucket solver defined


In [21]:
# ---------------------------------------------------------------------------
# Strategy 5: SPLIT QUEUE — explicit queue for nonstiff/mild, implicit for stiff.
# Classifies by stiffness proxy threshold at start.
# ---------------------------------------------------------------------------
def solve_split_queue_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    stiffness_threshold: float = 50.0,   # max |param| above this → implicit
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    B = len(y0_batch)
    stiff_proxy = np.max(np.abs(params_batch), axis=1)
    impl_mask   = stiff_proxy >= stiffness_threshold
    expl_mask   = ~impl_mask

    y_full     = np.zeros_like(y0_batch)
    n_acc_full = np.zeros(B, dtype=np.int32)
    n_rej_full = np.zeros(B, dtype=np.int32)
    n_nit_full = np.zeros(B, dtype=np.int32)

    combined = SolverStats()
    t_start = time.perf_counter()

    # Explicit sub-batch: override solver hint
    if np.any(expl_mask):
        ei = np.where(expl_mask)[0]
        import copy
        expl_problem = copy.copy(problem)
        expl_problem.solver_hint = 'explicit'
        yo, se = solve_compaction_jax(
            expl_problem, y0_batch[ei], params_batch[ei],
            rtol=rtol, atol=atol, h_init=h_init, h_min=h_min, h_max=h_max,
            newton_tol=newton_tol, newton_max_iter=newton_max_iter,
            max_iters=max_iters, safety=safety,
            min_factor=min_factor, max_factor=max_factor,
        )
        y_full[ei]     = yo
        n_acc_full[ei] = se.n_accept
        n_rej_full[ei] = se.n_reject
        n_nit_full[ei] = se.n_newton
        combined.n_steps_total += se.n_steps_total
        combined.active_fracs  += se.active_fracs
        combined.h_covs        += se.h_covs
        print(f'  Explicit queue: {len(ei)} trajectories, {se.n_steps_total} steps')

    # Implicit sub-batch
    if np.any(impl_mask) and problem.jac is not None:
        ii = np.where(impl_mask)[0]
        import copy
        impl_problem = copy.copy(problem)
        impl_problem.solver_hint = 'implicit'
        h0_impl = float(np.clip(0.1 / max(stiff_proxy[impl_mask].mean(), 1.), h_min, h_max))
        yo, si = solve_compaction_jax(
            impl_problem, y0_batch[ii], params_batch[ii],
            rtol=rtol, atol=atol, h_init=h0_impl, h_min=h_min, h_max=h_max,
            newton_tol=newton_tol, newton_max_iter=newton_max_iter,
            max_iters=max_iters, safety=safety,
            min_factor=min_factor, max_factor=max_factor,
        )
        y_full[ii]     = yo
        n_acc_full[ii] = si.n_accept
        n_rej_full[ii] = si.n_reject
        n_nit_full[ii] = si.n_newton
        combined.n_steps_total += si.n_steps_total
        combined.active_fracs  += si.active_fracs
        combined.h_covs        += si.h_covs
        print(f'  Implicit queue: {len(ii)} trajectories, {si.n_steps_total} steps')

    combined.wall_time = time.perf_counter() - t_start
    combined.n_accept  = n_acc_full
    combined.n_reject  = n_rej_full
    combined.n_newton  = n_nit_full
    return y_full, combined

print('Split-queue solver defined')

Split-queue solver defined


In [22]:
# ---------------------------------------------------------------------------
# Strategy 6: DYNAMIC_CLUSTER — periodic compaction + step-size sorting.
#
# Every `cluster_every` steps:
#   1. Flush finished trajectories (compaction, like REBATCH).
#   2. Sort remaining active trajectories by current step size h (ascending).
#
# Placing homogeneous-h trajectories in adjacent positions reduces:
#   • Newton convergence spread: similar h → similar (I - h*J) conditioning
#     → whole batch converges in fewer iterations together.
#   • Step-acceptance divergence: trajectories with similar h are likely to
#     accept/reject together, reducing masked-write overhead.
#   • Within-warp h-CoV: adjacent batch slots map to the same CUDA warp (32
#     threads); sorting minimises intra-warp step-size heterogeneity.
#
# New metric: `warp_h_covs` — mean CoV of h within each warp-aligned block,
# logged after each sort.  Compare to `h_covs` (global) to see the
# improvement sorting buys.
# ---------------------------------------------------------------------------
def solve_dynamic_cluster_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    cluster_every: int = 20,   # compact+sort every this many steps
    warp_size: int = 32,       # GPU warp size for per-warp CoV metric
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    t0, tf = problem.t_span
    rhs = problem.rhs
    use_implicit = (problem.solver_hint == 'implicit') and (problem.jac is not None)

    B           = len(y0_batch)
    y_full      = np.array(y0_batch, dtype=np.float64)
    p_full      = np.array(params_batch, dtype=np.float64)
    n_acc_full  = np.zeros(B, dtype=np.int32)
    n_rej_full  = np.zeros(B, dtype=np.int32)
    n_nit_full  = np.zeros(B, dtype=np.int32)
    active_idx  = np.arange(B)   # maps compacted position → original index

    y     = jnp.array(y_full)
    p     = jnp.array(p_full)
    t     = jnp.full(B, float(t0))
    h     = jnp.full(B, h_init)
    done  = jnp.zeros(B, dtype=bool)
    n_acc = jnp.zeros(B, jnp.int32)
    n_rej = jnp.zeros(B, jnp.int32)
    n_nit = jnp.zeros(B, jnp.int32)

    stats     = SolverStats()
    t_start   = time.perf_counter()
    t_compact = 0.0

    for step in range(max_iters):
        active   = ~done
        n_active = int(jnp.sum(active))
        if n_active == 0:
            break

        # Standard diagnostics
        stats.active_fracs.append(n_active / B)
        h_act = np.array(h)[np.array(active)]
        stats.h_covs.append(
            float(h_act.std() / max(h_act.mean(), 1e-300)) if len(h_act) > 1 else 0.
        )

        h_eff = jnp.clip(
            jnp.where(active, jnp.minimum(h, tf - t), h),
            h_min, h_max
        )

        if use_implicit:
            tn = t + h_eff
            yn, ok, ni = _newton_be(rhs, problem.jac, y, tn, h_eff, p,
                                    tol=newton_tol, max_iter=newton_max_iter)
            n_nit = n_nit + ni
            f_new = jax.vmap(rhs, in_axes=(0, 0, 0))(tn, yn, p)
            f_old = jax.vmap(rhs, in_axes=(0, 0, 0))(t,  y,  p)
            ee    = _err_norm(y, yn, h_eff[:, None] * (f_new - f_old) / 2., rtol, atol)
            acc   = ok & active & (ee <= 1.)
            t     = jnp.where(acc, tn, t)
            y     = jnp.where(acc[:, None], yn, y)
            se    = jnp.where(ee > 0, ee, 1e-300)
            fac     = jnp.clip(safety * se**(-0.5), min_factor, max_factor)
            fac_rej = jnp.clip(safety * se**(-0.5), min_factor, 1.0)
            h = jnp.where(active,
                          jnp.clip(jnp.where(acc, h_eff * fac, h_eff * fac_rej), h_min, h_max),
                          h)
        else:
            yc, ev = jax.vmap(
                lambda t_, y_, h_, p_: _rk45_step(rhs, t_, y_, h_, p_)
            )(t, y, h_eff, p)
            err = _err_norm(y, yc, ev, rtol, atol)
            acc = (err <= 1.) & active
            t   = jnp.where(acc, t + h_eff, t)
            y   = jnp.where(acc[:, None], yc, y)
            se  = jnp.where(err > 0, err, 1e-300)
            fa  = jnp.where(err == 0., max_factor,
                            jnp.clip(safety * se**(-0.2), min_factor, max_factor))
            fr  = jnp.clip(safety * se**(-0.2), min_factor, 1.)
            h   = jnp.where(active,
                            jnp.clip(jnp.where(acc, h_eff * fa, h_eff * fr), h_min, h_max),
                            h)

        n_acc = n_acc + acc.astype(jnp.int32)
        n_rej = n_rej + (~acc & active).astype(jnp.int32)
        done  = done | (t >= tf - 1e-12)
        stats.n_steps_total += 1

        # ── Periodic compaction + step-size sort ──────────────────────────
        if (step + 1) % cluster_every == 0:
            tc0      = time.perf_counter()
            done_np  = np.array(done)
            h_np     = np.array(h)
            y_np     = np.array(y)
            t_np     = np.array(t)
            p_np     = np.array(p)
            n_acc_np = np.array(n_acc)
            n_rej_np = np.array(n_rej)
            n_nit_np = np.array(n_nit)

            # 1. Flush finished trajectories back to full output arrays
            if np.any(done_np):
                fin_orig = active_idx[done_np]
                y_full[fin_orig]     = y_np[done_np]
                n_acc_full[fin_orig] = n_acc_np[done_np]
                n_rej_full[fin_orig] = n_rej_np[done_np]
                n_nit_full[fin_orig] = n_nit_np[done_np]
                keep        = ~done_np
                active_idx  = active_idx[keep]
                y_np        = y_np[keep]
                t_np        = t_np[keep]
                h_np        = h_np[keep]
                p_np        = p_np[keep]
                n_acc_np    = n_acc_np[keep]
                n_rej_np    = n_rej_np[keep]
                n_nit_np    = n_nit_np[keep]

            # 2. Sort remaining by step size (ascending) → homogeneous warps
            if len(h_np) > 1:
                order = np.argsort(h_np)   # stable sort, O(Bc log Bc)

                # Per-warp h-CoV after sorting (diagnostic metric)
                h_sorted  = h_np[order]
                warp_covs = []
                for w_start in range(0, len(h_sorted), warp_size):
                    w_h = h_sorted[w_start : w_start + warp_size]
                    if len(w_h) > 1:
                        warp_covs.append(
                            float(w_h.std() / max(w_h.mean(), 1e-300))
                        )
                if warp_covs:
                    stats.warp_h_covs.append(float(np.mean(warp_covs)))

                active_idx = active_idx[order]
                y_np       = y_np[order]
                t_np       = t_np[order]
                h_np       = h_np[order]
                p_np       = p_np[order]
                n_acc_np   = n_acc_np[order]
                n_rej_np   = n_rej_np[order]
                n_nit_np   = n_nit_np[order]

            # 3. Push sorted/compacted arrays back to device
            Bc    = len(active_idx)
            y     = jnp.array(y_np)
            p     = jnp.array(p_np)
            t     = jnp.array(t_np)
            h     = jnp.array(h_np)
            n_acc = jnp.array(n_acc_np)
            n_rej = jnp.array(n_rej_np)
            n_nit = jnp.array(n_nit_np)
            done  = jnp.zeros(Bc, dtype=bool)   # fresh mask for new compacted batch

            t_compact += time.perf_counter() - tc0

    # Flush any remaining active trajectories
    if len(active_idx) > 0:
        y_full[active_idx]     = np.array(y)
        n_acc_full[active_idx] = np.array(n_acc)
        n_rej_full[active_idx] = np.array(n_rej)
        n_nit_full[active_idx] = np.array(n_nit)

    stats.wall_time           = time.perf_counter() - t_start
    stats.compaction_overhead = t_compact
    stats.n_accept            = n_acc_full
    stats.n_reject            = n_rej_full
    stats.n_newton            = n_nit_full
    return y_full, stats

print('Dynamic-cluster solver defined')


Dynamic-cluster solver defined


## 5. Correctness Verification

In [23]:
# ---------------------------------------------------------------------------
# Quick correctness check on problems with known reference solutions
# ---------------------------------------------------------------------------
VERIFY_BATCH = 32
rng_v = np.random.default_rng(42)

print('Correctness check (baseline solver, batch=32):')
print(f'{"Problem":>8}  {"MaxErr":>12}  {"Wall(s)":>9}  {"MeanAcc":>9}')
print('-' * 50)

for name, prob in PROBLEMS.items():
    if prob.ref_solution is None:
        continue
    y0  = prob.ic_generator(rng_v, VERIFY_BATCH)
    par = prob.param_sampler(rng_v, VERIFY_BATCH)
    tf  = prob.t_span[1]

    h0 = 1e-4 if prob.stiffness_label in ('strong', 'extreme') else 1e-3
    y_out, st = solve_baseline_jax(
        prob, y0, par,
        rtol=1e-6, atol=1e-8, h_init=h0,
        max_iters=1_000_000,
    )
    y_ref = prob.ref_solution(tf, y0, par)
    max_err = float(np.max(np.abs(y_out - y_ref)))
    print(f'{name:>8}  {max_err:>12.3e}  {st.wall_time:>9.3f}  {st.n_accept.mean():>9.1f}')

Correctness check (baseline solver, batch=32):
 Problem        MaxErr    Wall(s)    MeanAcc
--------------------------------------------------
      NS     4.826e-05     18.675      146.5
      MS     5.640e-07      6.428       92.3
      SS     3.169e-06    113.605     4349.4
      LD     2.028e-04    170.387     6063.5
      HE     3.393e-04    188.533     3226.2
      AD     3.754e-07      3.316       33.5


## 6. Strategy Comparison Benchmark

In [24]:
# ---------------------------------------------------------------------------
# GPU utilization sampler -- polls nvidia-smi from a background thread
# ---------------------------------------------------------------------------
class GPUSampler:
    def __init__(self, interval=0.25):
        self.interval        = interval
        self.util_samples    = []   # [(elapsed_s, gpu_util_pct), ...]
        self.mem_mb_samples  = []   # [(elapsed_s, used_mb), ...]
        self._stop   = threading.Event()
        self._t0     = None
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._avail  = self._probe()

    def _probe(self):
        try:
            subprocess.check_output(
                ["nvidia-smi", "--query-gpu=utilization.gpu",
                 "--format=csv,noheader,nounits"],
                stderr=subprocess.DEVNULL, timeout=2)
            return True
        except Exception:
            return False

    def _loop(self):
        while not self._stop.wait(self.interval):
            if not self._avail:
                break
            try:
                raw = subprocess.check_output(
                    ["nvidia-smi",
                     "--query-gpu=utilization.gpu,memory.used",
                     "--format=csv,noheader,nounits"],
                    stderr=subprocess.DEVNULL, timeout=1
                ).decode().strip()
                util, mem = raw.split(",")
                ts = time.perf_counter() - self._t0
                self.util_samples.append((ts, float(util.strip())))
                self.mem_mb_samples.append((ts, float(mem.strip())))
            except Exception:
                pass

    def start(self):
        self._t0 = time.perf_counter()
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()
        if self._thread.is_alive():
            self._thread.join(timeout=2)

print("GPUSampler defined")

# ---------------------------------------------------------------------------
# Benchmark setup — shared across all per-problem cells below
# ---------------------------------------------------------------------------
BENCH_PROBLEMS = ['MS', 'SS', 'MX', 'VZ', 'HE', 'AD']
BATCH_SIZE     = 256

# Resume: load any previously completed results from Drive
_ckpt = load_from_drive(CKPT_MAIN)
all_results = _ckpt if _ckpt is not None else {}
if all_results:
    print(f'[resume] Loaded {len(all_results)} completed result(s) from Drive.')

STRATEGIES = {
    'baseline':   lambda p, y0, par: solve_baseline_jax(p, y0, par, max_iters=500_000),
    'compaction': lambda p, y0, par: solve_compaction_jax(p, y0, par, max_iters=500_000),
    'rebatch_50': lambda p, y0, par: solve_rebatch_jax(p, y0, par, rebatch_every=50, max_iters=500_000),
    'rebatch_10': lambda p, y0, par: solve_rebatch_jax(p, y0, par, rebatch_every=10, max_iters=500_000),
    'buckets_4':  lambda p, y0, par: solve_buckets_jax(p, y0, par, n_buckets=4, max_iters=500_000),
    'split_q':     lambda p, y0, par: solve_split_queue_jax(p, y0, par, max_iters=500_000),
    'dyn_cluster': lambda p, y0, par: solve_dynamic_cluster_jax(p, y0, par, cluster_every=20, max_iters=500_000),
}

def _run_problem_bench(pname, batch_size=BATCH_SIZE, seed=None, strategy_overrides=None):
    """Run all strategies on one problem; skip completed pairs; save after each."""
    prob = PROBLEMS[pname]
    rng  = np.random.default_rng(seed if seed is not None else hash(pname) % (2**31))
    y0   = prob.ic_generator(rng, batch_size)
    par  = prob.param_sampler(rng, batch_size)
    strats = strategy_overrides if strategy_overrides is not None else STRATEGIES
    print(f'\n=== {pname} ({prob.stiffness_label}, dim={prob.state_dim}, '
          f't∈{prob.t_span}, B={batch_size}) ===')
    for sname, solver_fn in strats.items():
        key = (pname, sname)
        if key in all_results:
            s = all_results[key].summary()
            print(f'  {sname:>14}: [resumed] wall={s["wall_time"]:.3f}s')
            continue
        try:
            _sampler = GPUSampler(interval=0.20).start()
            y_out, st = solver_fn(prob, y0, par)
            _sampler.stop()
            st.gpu_util_samples   = _sampler.util_samples
            st.gpu_mem_mb_samples = _sampler.mem_mb_samples
            st.batch_size         = batch_size
            all_results[key] = st
            save_to_drive(all_results, CKPT_MAIN)
            s = st.summary()
            print(f'  {sname:>14}: wall={s["wall_time"]:.3f}s  '
                  f'tput={s["throughput"]:.1f}/s  '
                  f'active={s["mean_active_frac"]:.1%}  '
                  f'h_cov={s["mean_h_cov"]:.3f}  rej={s["rej_rate"]:.1%}  '
                  f'newton={s["mean_newton"]:.1f}  '
                  f'gpu={s["mean_gpu_util"]:.0f}%  '
                  f'warp_hcov={s["mean_warp_h_cov"]:.4f}')
        except Exception as e:
            print(f'  {sname:>14}: FAILED — {e}')

# ----- MS: Mildly stiff (λ∈[1,10], t_end=6) -----
_run_problem_bench('MS', seed=1337)


GPUSampler defined

=== MS (mild, dim=4, t∈(0.0, 6.0), B=256) ===
  [ckpt] saved → all_results.pkl
        baseline: wall=9.218s  tput=27.8/s  active=81.6%  h_cov=0.143  rej=2.3%  newton=0.0  gpu=0%  warp_hcov=0.0000
  [ckpt] saved → all_results.pkl
      compaction: wall=117.799s  tput=2.2/s  active=81.6%  h_cov=0.143  rej=2.3%  newton=0.0  gpu=0%  warp_hcov=0.0000
  [ckpt] saved → all_results.pkl
      rebatch_50: wall=7.272s  tput=35.2/s  active=81.6%  h_cov=0.143  rej=2.3%  newton=0.0  gpu=0%  warp_hcov=0.0000
  [ckpt] saved → all_results.pkl
      rebatch_10: wall=6.796s  tput=37.7/s  active=81.6%  h_cov=0.143  rej=2.3%  newton=0.0  gpu=0%  warp_hcov=0.0000
  [ckpt] saved → all_results.pkl
       buckets_4: wall=22.868s  tput=11.2/s  active=90.7%  h_cov=0.097  rej=2.3%  newton=0.0  gpu=0%  warp_hcov=0.0000
  Explicit queue: 256 trajectories, 114 steps
  [ckpt] saved → all_results.pkl
         split_q: wall=5.196s  tput=49.3/s  active=81.6%  h_cov=0.143  rej=2.3%  newton=0.0  gpu=0

In [ ]:
# ----- SS: Strongly stiff (λ∈[100,2000], t_end=1) -----
# Explicit RK45 must use tiny steps (h~1/2000); implicit BE trivially stable.
_run_problem_bench('SS', seed=2001)



=== SS (strong, dim=4, t∈(0.0, 1.0), B=256) ===
  [ckpt] saved → all_results.pkl
        baseline: wall=134.857s  tput=1.9/s  active=68.2%  h_cov=1.938  rej=0.1%  newton=12396.0  gpu=0%  warp_hcov=0.0000


In [ ]:
# ----- MX: Mixed stiffness (fast λ∈[500,1000] + slow λ∈[0.5,5], ε=0.01, t_end=2) -----
# Tests h-spread between fast/slow modes within the same trajectory.
_run_problem_bench('MX', seed=3001)


In [ ]:
# ----- VZ: Van der Pol (μ∈[1,50], t_end=10) -----
# Stiffness O(μ²)~2500. Use B=128 and max_iters=200_000 to bound runtime.
_vz_overrides = {
    'baseline':   lambda p, y0, par: solve_baseline_jax(p, y0, par, max_iters=200_000),
    'compaction': lambda p, y0, par: solve_compaction_jax(p, y0, par, max_iters=200_000),
    'rebatch_50': lambda p, y0, par: solve_rebatch_jax(p, y0, par, rebatch_every=50, max_iters=200_000),
    'rebatch_10': lambda p, y0, par: solve_rebatch_jax(p, y0, par, rebatch_every=10, max_iters=200_000),
    'buckets_4':  lambda p, y0, par: solve_buckets_jax(p, y0, par, n_buckets=4, max_iters=200_000),
    'split_q':    lambda p, y0, par: solve_split_queue_jax(p, y0, par, max_iters=200_000),
}
_run_problem_bench('VZ', batch_size=128, seed=4001, strategy_overrides=_vz_overrides)


In [ ]:
# ----- HE: 1D Heat equation MoL (PDE parabolic, d=20, α∈[0.1,2], t_end=0.5) -----
# Tridiagonal 20×20 Jacobian — exercises batched jnp.linalg.solve on GPU.
# Step-size spread scales with α; trajectories with large α need tiny h.
_run_problem_bench('HE', seed=5001)


In [ ]:
# ----- AD: 1D Advection MoL (PDE hyperbolic, d=20, c∈[0.2,2], t_end=1) -----
# Explicit only (nonstiff). CFL h_stable ∝ 1/c → 10× step-size spread.
# Best case for COMPACTION and BUCKETS: done trajectories (high c) finish early.
_run_problem_bench('AD', seed=6001)


## 7. Large Batch Scaling

In [ ]:
# ---------------------------------------------------------------------------
# Scaling study: baseline vs compaction on SS — small batches [64, 128, 256]
# ---------------------------------------------------------------------------
SCALE_PROBLEM = 'SS'
prob_s  = PROBLEMS[SCALE_PROBLEM]
rng_s   = np.random.default_rng(999)

# Resume: load existing scale records
_scale_ckpt = load_from_drive(CKPT_SCALE)
scale_records = _scale_ckpt if _scale_ckpt is not None else []
_done_bs = {r['bs'] for r in scale_records}
if _done_bs:
    print(f'[resume] Loaded scale records for batches: {sorted(_done_bs)}')

print(f'Scaling study on {SCALE_PROBLEM} — small batches:')
print(f'{"Batch":>6} {"Baseline(s)":>13} {"Compact(s)":>12} {"Speedup":>9} '
      f'{"ActiveFrac":>12} {"hCoV":>8}')
print('-'*65)

for bs in [64, 128, 256]:
    if bs in _done_bs:
        r = next(r for r in scale_records if r['bs'] == bs)
        print(f'{bs:>6} {r["t_base"]:>13.3f} {r["t_comp"]:>12.3f} '
              f'{r["speedup"]:>9.2f}x {r["af_comp"]:>12.1%} '
              f'{r["hcov_comp"]:>8.3f}  [resumed]')
        continue
    y0  = prob_s.ic_generator(rng_s, bs)
    par = prob_s.param_sampler(rng_s, bs)
    _, st_b = solve_baseline_jax(prob_s, y0, par, max_iters=200_000)
    _, st_c = solve_compaction_jax(prob_s, y0, par, max_iters=200_000)
    sb = st_b.summary(); sc = st_c.summary()
    speedup = sb['wall_time'] / max(sc['wall_time'], 1e-9)
    rec = dict(bs=bs, t_base=sb['wall_time'], t_comp=sc['wall_time'], speedup=speedup,
               af_base=sb['mean_active_frac'], af_comp=sc['mean_active_frac'],
               hcov_base=sb['mean_h_cov'],    hcov_comp=sc['mean_h_cov'])
    scale_records.append(rec)
    save_to_drive(scale_records, CKPT_SCALE)
    _done_bs.add(bs)
    print(f'{bs:>6} {sb["wall_time"]:>13.3f} {sc["wall_time"]:>12.3f} '
          f'{speedup:>9.2f}x {sc["mean_active_frac"]:>12.1%} '
          f'{sc["mean_h_cov"]:>8.3f}')


In [ ]:
# ---------------------------------------------------------------------------
# Scaling study continued — large batches [512, 1024]
# (split to avoid >30 min per cell on CPU; GPU runs these quickly)
# ---------------------------------------------------------------------------
rng_sl = np.random.default_rng(9999)
_done_bs_large = {r['bs'] for r in scale_records}

print(f'Scaling study on {SCALE_PROBLEM} — large batches:')
print(f'{"Batch":>6} {"Baseline(s)":>13} {"Compact(s)":>12} {"Speedup":>9} '
      f'{"ActiveFrac":>12} {"hCoV":>8}')
print('-'*65)

for bs in [512, 1024]:
    if bs in _done_bs_large:
        r = next(r for r in scale_records if r['bs'] == bs)
        print(f'{bs:>6} {r["t_base"]:>13.3f} {r["t_comp"]:>12.3f} '
              f'{r["speedup"]:>9.2f}x {r["af_comp"]:>12.1%} '
              f'{r["hcov_comp"]:>8.3f}  [resumed]')
        continue
    y0  = prob_s.ic_generator(rng_sl, bs)
    par = prob_s.param_sampler(rng_sl, bs)
    _, st_b = solve_baseline_jax(prob_s, y0, par, max_iters=200_000)
    _, st_c = solve_compaction_jax(prob_s, y0, par, max_iters=200_000)
    sb = st_b.summary(); sc = st_c.summary()
    speedup = sb['wall_time'] / max(sc['wall_time'], 1e-9)
    rec = dict(bs=bs, t_base=sb['wall_time'], t_comp=sc['wall_time'], speedup=speedup,
               af_base=sb['mean_active_frac'], af_comp=sc['mean_active_frac'],
               hcov_base=sb['mean_h_cov'],    hcov_comp=sc['mean_h_cov'])
    scale_records.append(rec)
    save_to_drive(scale_records, CKPT_SCALE)
    _done_bs_large.add(bs)
    print(f'{bs:>6} {sb["wall_time"]:>13.3f} {sc["wall_time"]:>12.3f} '
          f'{speedup:>9.2f}x {sc["mean_active_frac"]:>12.1%} '
          f'{sc["mean_h_cov"]:>8.3f}')


## 8. Large-Dimension Test (LD)

In [ ]:
# ---------------------------------------------------------------------------
# Large-dim (d=64) benchmark: GPU register pressure + batched linear solve
# ---------------------------------------------------------------------------
prob_ld = PROBLEMS['LD']
LD_BATCHES = [16, 32, 64, 128]
rng_ld = np.random.default_rng(7777)

# Resume: load existing LD records
_ld_ckpt  = load_from_drive(CKPT_LD)
ld_records = _ld_ckpt if _ld_ckpt is not None else {}
if ld_records:
    print(f'[resume] Loaded LD records for batches: {sorted(ld_records.keys())}')

print(f'Large-dimension benchmark (d={prob_ld.state_dim}):')
print(f'{"Batch":>6} {"Baseline(s)":>13} {"Compact(s)":>12} '
      f'{"MaxErr(base)":>14} {"MaxErr(comp)":>14}')
print('-'*65)

for bs in LD_BATCHES:
    if bs in ld_records:
        r = ld_records[bs]
        print(f'{bs:>6} {r["t_base"]:>13.3f} {r["t_comp"]:>12.3f} '
              f'{r["err_b"]:>14.3e} {r["err_c"]:>14.3e}  [resumed]')
        continue
    y0  = prob_ld.ic_generator(rng_ld, bs)
    par = prob_ld.param_sampler(rng_ld, bs)
    tf  = prob_ld.t_span[1]

    y_b, st_b = solve_baseline_jax(prob_ld, y0, par, h_init=1e-4, max_iters=200_000)
    y_c, st_c = solve_compaction_jax(prob_ld, y0, par, h_init=1e-4, max_iters=200_000)
    y_ref = prob_ld.ref_solution(tf, y0, par)
    err_b = float(np.max(np.abs(y_b - y_ref)))
    err_c = float(np.max(np.abs(y_c - y_ref)))
    ld_records[bs] = dict(t_base=st_b.wall_time, t_comp=st_c.wall_time,
                          err_b=err_b, err_c=err_c)
    save_to_drive(ld_records, CKPT_LD)
    print(f'{bs:>6} {st_b.wall_time:>13.3f} {st_c.wall_time:>12.3f} '
          f'{err_b:>14.3e} {err_c:>14.3e}')


## 9. Visualization

In [ ]:
# ---------------------------------------------------------------------------
# Plot 1: Time-series divergence diagnostics
#   Row 0: active fraction + h-CoV over iterations (SS problem)
#   Row 1: GPU utilization over time + warp h-CoV improvement (dyn_cluster)
# ---------------------------------------------------------------------------
_REF_PROBLEM = "SS" if "SS" in BENCH_PROBLEMS else BENCH_PROBLEMS[0]
_ALL_STRATS  = list(STRATEGIES.keys())
_CMAP = plt.cm.get_cmap("tab10", len(_ALL_STRATS))
_COLORS = {s: _CMAP(i) for i, s in enumerate(_ALL_STRATS)}

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle(f"GPU Divergence Time-Series — {_REF_PROBLEM} problem, B={BATCH_SIZE}", fontsize=13)
ax_af, ax_hc, ax_gpu, ax_warp = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

for sname in _ALL_STRATS:
    key = (_REF_PROBLEM, sname)
    if key not in all_results:
        continue
    st  = all_results[key]
    s   = st.summary()
    col = _COLORS[sname]
    lbl = f"{sname} ({s["wall_time"]:.2f}s)"

    af  = np.array(st.active_fracs)
    hc  = np.array(st.h_covs)
    xs  = np.arange(len(af))
    ax_af.plot(xs, af, color=col, alpha=0.8, label=lbl, lw=1.2)
    ax_hc.plot(xs, hc, color=col, alpha=0.8, label=lbl, lw=1.2)

    # GPU utilization time series (wall-clock seconds)
    if st.gpu_util_samples:
        ts_gpu  = np.array([t for t, _ in st.gpu_util_samples])
        util_pct = np.array([u for _, u in st.gpu_util_samples])
        ax_gpu.plot(ts_gpu, util_pct, color=col, alpha=0.8,
                    label=f"{sname} (mean {s["mean_gpu_util"]:.0f}%)", lw=1.2)

ax_af.set_xlabel("Iteration"); ax_af.set_ylabel("Active fraction")
ax_af.set_title("Active Fraction over Iterations")
ax_af.legend(fontsize=7, ncol=2); ax_af.grid(True, alpha=0.3)

ax_hc.set_xlabel("Iteration"); ax_hc.set_ylabel("Step-size CoV")
ax_hc.set_title("Step-size Spread (h-CoV) over Iterations")
ax_hc.legend(fontsize=7, ncol=2); ax_hc.grid(True, alpha=0.3)

if ax_gpu.lines:
    ax_gpu.set_xlabel("Wall-clock time (s)"); ax_gpu.set_ylabel("GPU utilization (%)")
    ax_gpu.set_title("GPU Utilization over Time")
    ax_gpu.set_ylim(0, 105); ax_gpu.legend(fontsize=7, ncol=2)
    ax_gpu.grid(True, alpha=0.3)
else:
    ax_gpu.text(0.5, 0.5, "GPU util not available\n(run on NVIDIA GPU / Colab)",
               ha="center", va="center", transform=ax_gpu.transAxes, fontsize=11,
               bbox=dict(boxstyle="round", facecolor="lightyellow"))
    ax_gpu.set_title("GPU Utilization over Time")
    ax_gpu.axis("off")

# Warp h-CoV comparison: global h-CoV vs within-warp h-CoV for dyn_cluster
_dc_key = (_REF_PROBLEM, "dyn_cluster")
if _dc_key in all_results and all_results[_dc_key].warp_h_covs:
    _st = all_results[_dc_key]
    _hc_global = np.array(_st.h_covs)
    _warp_cov  = np.array(_st.warp_h_covs)
    # Downsample global h_cov to match cluster events
    _cluster_every = 20
    _sample_idx    = np.arange(len(_warp_cov)) * _cluster_every
    _sample_idx    = _sample_idx[_sample_idx < len(_hc_global)]
    ax_warp.plot(np.arange(len(_warp_cov[:len(_sample_idx)])),
                 _warp_cov[:len(_sample_idx)],
                 "o-", color="crimson", lw=1.5, ms=4, label="warp h-CoV (post-sort)")
    ax_warp.plot(np.arange(len(_sample_idx)),
                 _hc_global[_sample_idx],
                 "s--", color="steelblue", lw=1.5, ms=4, label="global h-CoV at cluster")
    ax_warp.set_xlabel("Cluster event #"); ax_warp.set_ylabel("h-CoV")
    ax_warp.set_title("dyn_cluster: Warp h-CoV vs Global h-CoV\n(lower = more homogeneous warps)")
    ax_warp.legend(fontsize=9); ax_warp.grid(True, alpha=0.3)
else:
    # Fallback: bar chart of mean warp h-CoV across all strategies
    _names = [s for s in _ALL_STRATS if (_REF_PROBLEM, s) in all_results]
    _vals  = [all_results[(_REF_PROBLEM, s)].summary()["mean_warp_h_cov"] for s in _names]
    ax_warp.bar(_names, _vals, color=[_COLORS[s] for s in _names], alpha=0.8)
    ax_warp.set_ylabel("Mean warp h-CoV"); ax_warp.set_title("Mean Warp h-CoV by Strategy")
    ax_warp.tick_params(axis="x", rotation=25); ax_warp.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
_f1 = "gpu_bench_timeseries.png"
plt.savefig(_f1, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved {_f1}")
if IN_COLAB:
    import shutil; shutil.copy(_f1, os.path.join(DRIVE_DIR, _f1))


In [ ]:
# ---------------------------------------------------------------------------
# Plot 2: Performance comparison — wall time, throughput, GPU util, active frac
# ---------------------------------------------------------------------------
_ALL_STRATS = list(STRATEGIES.keys())
_CMAP2 = plt.cm.get_cmap("tab10", len(_ALL_STRATS))
_C2    = {s: _CMAP2(i) for i, s in enumerate(_ALL_STRATS)}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f"GPU Strategy Performance Comparison — B={BATCH_SIZE}", fontsize=13)
_x  = np.arange(len(BENCH_PROBLEMS))
_w  = 0.85 / max(len(_ALL_STRATS), 1)

# 2a: Wall time
ax = axes[0,0]
for i, sname in enumerate(_ALL_STRATS):
    vals = [all_results[(_p, sname)].summary()["wall_time"]
            if (_p, sname) in all_results else 0.
            for _p in BENCH_PROBLEMS]
    ax.bar(_x + i * _w, vals, _w, label=sname, color=_C2[sname], alpha=0.85)
ax.set_xticks(_x + _w * (len(_ALL_STRATS) - 1) / 2)
ax.set_xticklabels(BENCH_PROBLEMS)
ax.set_ylabel("Wall time (s)"); ax.set_title("Wall Time per Strategy & Problem")
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3, axis="y")

# 2b: Throughput (trajectories/s)
ax = axes[0,1]
for i, sname in enumerate(_ALL_STRATS):
    vals = [all_results[(_p, sname)].summary()["throughput"]
            if (_p, sname) in all_results else 0.
            for _p in BENCH_PROBLEMS]
    ax.bar(_x + i * _w, vals, _w, label=sname, color=_C2[sname], alpha=0.85)
ax.set_xticks(_x + _w * (len(_ALL_STRATS) - 1) / 2)
ax.set_xticklabels(BENCH_PROBLEMS)
ax.set_ylabel("Trajectories / second"); ax.set_title("Throughput per Strategy & Problem")
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3, axis="y")

# 2c: Mean GPU utilization (%)
ax = axes[1,0]
_has_gpu = any(
    all_results[k].gpu_util_samples
    for k in all_results if isinstance(k, tuple)
)
if _has_gpu:
    for i, sname in enumerate(_ALL_STRATS):
        vals = [all_results[(_p, sname)].summary()["mean_gpu_util"]
                if (_p, sname) in all_results else 0.
                for _p in BENCH_PROBLEMS]
        ax.bar(_x + i * _w, vals, _w, label=sname, color=_C2[sname], alpha=0.85)
    ax.set_xticks(_x + _w * (len(_ALL_STRATS) - 1) / 2)
    ax.set_xticklabels(BENCH_PROBLEMS)
    ax.set_ylabel("Mean GPU utilization (%)"); ax.set_ylim(0, 105)
    ax.set_title("Mean GPU Utilization per Strategy & Problem")
    ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3, axis="y")
else:
    ax.text(0.5, 0.5, "GPU utilization data not available\n(requires NVIDIA GPU + nvidia-smi)",
            ha="center", va="center", transform=ax.transAxes, fontsize=11,
            bbox=dict(boxstyle="round", facecolor="lightyellow"))
    ax.set_title("Mean GPU Utilization per Strategy & Problem")
    ax.axis("off")

# 2d: Mean active fraction
ax = axes[1,1]
for i, sname in enumerate(_ALL_STRATS):
    vals = [all_results[(_p, sname)].summary()["mean_active_frac"]
            if (_p, sname) in all_results else 0.
            for _p in BENCH_PROBLEMS]
    ax.bar(_x + i * _w, vals, _w, label=sname, color=_C2[sname], alpha=0.85)
ax.set_xticks(_x + _w * (len(_ALL_STRATS) - 1) / 2)
ax.set_xticklabels(BENCH_PROBLEMS)
ax.set_ylabel("Mean active fraction"); ax.set_ylim(0, 1.05)
ax.set_title("Mean Active Fraction per Strategy & Problem")
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
_f2 = "gpu_bench_performance.png"
plt.savefig(_f2, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved {_f2}")
if IN_COLAB:
    import shutil; shutil.copy(_f2, os.path.join(DRIVE_DIR, _f2))


In [ ]:
# ---------------------------------------------------------------------------
# Plot 3: Overhead analysis + quality metrics
#   Row 0: compaction overhead %, rejection rate
#   Row 1: per-problem active-fraction decay (baseline vs dyn_cluster)
# ---------------------------------------------------------------------------
_ALL_STRATS = list(STRATEGIES.keys())
_CMAP3 = plt.cm.get_cmap("tab10", len(_ALL_STRATS))
_C3    = {s: _CMAP3(i) for i, s in enumerate(_ALL_STRATS)}
_x  = np.arange(len(BENCH_PROBLEMS))
_w  = 0.85 / max(len(_ALL_STRATS), 1)

_n_bench = len(BENCH_PROBLEMS)
_n_cols  = 3
_n_rows  = ((_n_bench + _n_cols - 1) // _n_cols) + 2
fig = plt.figure(figsize=(15, 4.5 * _n_rows))
import matplotlib.gridspec as _gs
_spec = _gs.GridSpec(_n_rows, _n_cols, figure=fig)

# Row 0 left (span 2 cols): compaction overhead % of wall time
ax_oh = fig.add_subplot(_spec[0, :2])
for i, sname in enumerate(_ALL_STRATS):
    oh_pcts = []
    for _p in BENCH_PROBLEMS:
        k = (_p, sname)
        if k in all_results:
            s = all_results[k].summary()
            oh_pcts.append(100 * s["compaction_overhead"] / max(s["wall_time"], 1e-9))
        else:
            oh_pcts.append(0.)
    ax_oh.bar(_x + i * _w, oh_pcts, _w, label=sname, color=_C3[sname], alpha=0.85)
ax_oh.set_xticks(_x + _w * (len(_ALL_STRATS) - 1) / 2)
ax_oh.set_xticklabels(BENCH_PROBLEMS)
ax_oh.set_ylabel("Overhead (% wall time)")
ax_oh.set_title("Compaction Overhead as % of Wall Time")
ax_oh.legend(fontsize=7, ncol=2); ax_oh.grid(True, alpha=0.3, axis="y")

# Row 0 right: rejection rate
ax_rej = fig.add_subplot(_spec[0, 2])
for i, sname in enumerate(_ALL_STRATS):
    rr = [all_results[(_p, sname)].summary()["rej_rate"]
          if (_p, sname) in all_results else 0.
          for _p in BENCH_PROBLEMS]
    ax_rej.bar(_x + i * _w, rr, _w, label=sname, color=_C3[sname], alpha=0.85)
ax_rej.set_xticks(_x + _w * (len(_ALL_STRATS) - 1) / 2)
ax_rej.set_xticklabels(BENCH_PROBLEMS)
ax_rej.set_ylabel("Rejection rate"); ax_rej.set_title("Step Rejection Rate")
ax_rej.legend(fontsize=6, ncol=2); ax_rej.grid(True, alpha=0.3, axis="y")

# Row 1 left: mean h-CoV
ax_hcov = fig.add_subplot(_spec[1, :2])
for i, sname in enumerate(_ALL_STRATS):
    hcv = [all_results[(_p, sname)].summary()["mean_h_cov"]
           if (_p, sname) in all_results else 0.
           for _p in BENCH_PROBLEMS]
    ax_hcov.bar(_x + i * _w, hcv, _w, label=sname, color=_C3[sname], alpha=0.85)
ax_hcov.set_xticks(_x + _w * (len(_ALL_STRATS) - 1) / 2)
ax_hcov.set_xticklabels(BENCH_PROBLEMS)
ax_hcov.set_ylabel("Mean h-CoV"); ax_hcov.set_title("Step-size Spread (lower = more uniform steps)")
ax_hcov.legend(fontsize=7, ncol=2); ax_hcov.grid(True, alpha=0.3, axis="y")

# Row 1 right: mean Newton iters
ax_nwt = fig.add_subplot(_spec[1, 2])
for i, sname in enumerate(_ALL_STRATS):
    nw = [all_results[(_p, sname)].summary()["mean_newton"]
          if (_p, sname) in all_results else 0.
          for _p in BENCH_PROBLEMS]
    ax_nwt.bar(_x + i * _w, nw, _w, label=sname, color=_C3[sname], alpha=0.85)
ax_nwt.set_xticks(_x + _w * (len(_ALL_STRATS) - 1) / 2)
ax_nwt.set_xticklabels(BENCH_PROBLEMS)
ax_nwt.set_ylabel("Mean Newton iters/traj"); ax_nwt.set_title("Newton Iterations")
ax_nwt.legend(fontsize=6, ncol=2); ax_nwt.grid(True, alpha=0.3, axis="y")

# Rows 2+: per-problem active fraction decay (baseline vs dyn_cluster)
_cmp_strats = [(s, c) for s, c in [("baseline", "steelblue"), ("dyn_cluster", "crimson")]
               if any((p, s) in all_results for p in BENCH_PROBLEMS)]
for idx, pname in enumerate(BENCH_PROBLEMS):
    row = 2 + idx // _n_cols
    col = idx % _n_cols
    ax = fig.add_subplot(_spec[row, col])
    for sname, col_c in _cmp_strats:
        k = (pname, sname)
        if k not in all_results:
            continue
        st = all_results[k]
        af = np.array(st.active_fracs)
        ax.plot(np.arange(len(af)), af, lw=1.2, color=col_c,
                label=f"{sname} (wasted {1-np.mean(af):.1%})")
        ax.fill_between(np.arange(len(af)), af, alpha=0.1, color=col_c)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Iteration"); ax.set_ylabel("Active fraction")
    prob = PROBLEMS[pname]
    ax.set_title(f"{pname} ({prob.stiffness_label})")
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(_n_bench, ((_n_rows - 2) * _n_cols)):
    row = 2 + idx // _n_cols
    col = idx % _n_cols
    if row < _n_rows:
        try:
            fig.add_subplot(_spec[row, col]).set_visible(False)
        except Exception:
            pass

plt.tight_layout()
_f3 = "gpu_bench_overhead.png"
plt.savefig(_f3, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved {_f3}")
if IN_COLAB:
    import shutil; shutil.copy(_f3, os.path.join(DRIVE_DIR, _f3))


In [ ]:
# ---------------------------------------------------------------------------
# Plot 4: Scaling study + speedup curves (all strategies)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Scaling Study on {SCALE_PROBLEM} problem", fontsize=13)
_ALL_STRATS = list(STRATEGIES.keys())
_CMAP4 = plt.cm.get_cmap("tab10", len(_ALL_STRATS))
_C4    = {s: _CMAP4(i) for i, s in enumerate(_ALL_STRATS)}

# Left: scaling curves
ax = axes[0]
_bs_arr = np.array([r["bs"] for r in scale_records])
_t_base = np.array([r["t_base"] for r in scale_records])
_t_comp = np.array([r["t_comp"] for r in scale_records])
ax.loglog(_bs_arr, _t_base, "-o", label="baseline",   color=_C4["baseline"])
ax.loglog(_bs_arr, _t_comp, "-s", label="compaction", color=_C4["compaction"])
ax.set_xlabel("Batch size"); ax.set_ylabel("Wall time (s)")
ax.set_title(f"Scaling: baseline vs compaction")
ax.legend(); ax.grid(True, which="both", alpha=0.3)

# Right: normalized strategy comparison heatmap across problems
ax = axes[1]
_metrics   = ["wall_time", "mean_active_frac", "mean_h_cov", "rej_rate",
               "mean_newton", "compaction_overhead", "mean_gpu_util", "mean_warp_h_cov"]
_mlabels   = ["Wall time", "Active frac", "h-CoV", "Rej rate",
               "Newton/traj", "Compact OH", "GPU util %", "Warp h-CoV"]
_strat_list = _ALL_STRATS

# Build matrix: rows=strategies, cols=metrics, value=mean over problems (normalized 0-1)
_mat = np.zeros((len(_strat_list), len(_metrics)))
for si, sn in enumerate(_strat_list):
    for mi, mk in enumerate(_metrics):
        vals = [all_results[(_p, sn)].summary()[mk]
                for _p in BENCH_PROBLEMS if (_p, sn) in all_results]
        _mat[si, mi] = np.mean(vals) if vals else np.nan

# Normalize each column to [0,1] (ignoring NaN)
_mat_norm = np.zeros_like(_mat)
for mi in range(len(_metrics)):
    col = _mat[:, mi]
    _lo, _hi = np.nanmin(col), np.nanmax(col)
    if _hi > _lo:
        _mat_norm[:, mi] = (col - _lo) / (_hi - _lo)
    else:
        _mat_norm[:, mi] = 0.5

_im = ax.imshow(_mat_norm, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(_metrics)))
ax.set_yticks(np.arange(len(_strat_list)))
ax.set_xticklabels(_mlabels, rotation=35, ha="right", fontsize=8)
ax.set_yticklabels(_strat_list, fontsize=9)
ax.set_title("Strategy × Metric Heatmap\n(red=worst, green=best, normalized per column)")
plt.colorbar(_im, ax=ax, fraction=0.035, pad=0.04)

# Annotate cells with actual mean values
for si in range(len(_strat_list)):
    for mi in range(len(_metrics)):
        v = _mat[si, mi]
        if not np.isnan(v):
            txt = f"{v:.2f}" if v < 100 else f"{v:.0f}"
            ax.text(mi, si, txt, ha="center", va="center", fontsize=6,
                    color="black")

plt.tight_layout()
_f4 = "gpu_bench_heatmap.png"
plt.savefig(_f4, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved {_f4}")
if IN_COLAB:
    import shutil; shutil.copy(_f4, os.path.join(DRIVE_DIR, _f4))


In [ ]:
# ---------------------------------------------------------------------------
# Final summary table — all strategies, all problems, all metrics
# ---------------------------------------------------------------------------
_ALL_STRATS = list(STRATEGIES.keys())
_HDR = (f'{"Problem":>8} {"Strategy":>14} {"Wall(s)":>8} {"Tput/s":>8} '
        f'{"GPU%":>6} {"PkMem":>7} {"Active":>8} {"hCoV":>7} '
        f'{"WarpCoV":>8} {"RejRate":>8} {"Newton":>7} {"CmpOH":>7}')
print()
print("=" * len(_HDR))
print(_HDR)
print("=" * len(_HDR))
for pname in BENCH_PROBLEMS:
    for sname in _ALL_STRATS:
        key = (pname, sname)
        if key not in all_results:
            continue
        s = all_results[key].summary()
        print(f'{pname:>8} {sname:>14} '
              f'{s["wall_time"]:>8.3f} '
              f'{s["throughput"]:>8.1f} '
              f'{s["mean_gpu_util"]:>6.1f} '
              f'{s["peak_gpu_mem_mb"]:>7.0f} '
              f'{s["mean_active_frac"]:>8.2%} '
              f'{s["mean_h_cov"]:>7.3f} '
              f'{s["mean_warp_h_cov"]:>8.4f} '
              f'{s["rej_rate"]:>8.2%} '
              f'{s["mean_newton"]:>7.2f} '
              f'{s["compaction_overhead"]:>7.3f}')
    print("-" * len(_HDR))
print("=" * len(_HDR))

# Save complete CSV
_fields = ["problem", "strategy", "wall_time", "throughput",
           "mean_gpu_util", "min_gpu_util", "peak_gpu_mem_mb",
           "mean_active_frac", "min_active_frac",
           "mean_h_cov", "max_h_cov", "mean_warp_h_cov",
           "rej_rate", "mean_newton", "compaction_overhead", "n_steps_total"]
_csv_rows = []
for pname in BENCH_PROBLEMS:
    for sname in _ALL_STRATS:
        k = (pname, sname)
        if k not in all_results:
            continue
        s = all_results[k].summary()
        row = {"problem": pname, "strategy": sname}
        row.update({f: round(s[f], 6) for f in _fields if f not in ("problem", "strategy")})
        _csv_rows.append(row)

import csv as _csv, io as _io
_buf = _io.StringIO()
_w = _csv.DictWriter(_buf, fieldnames=_fields)
_w.writeheader(); _w.writerows(_csv_rows)
_csv_str = _buf.getvalue()

if IN_COLAB:
    _csv_path = os.path.join(DRIVE_DIR, "bench_summary.csv")
    with open(_csv_path, "w") as _f:
        _f.write(_csv_str)
    print(f"Summary CSV saved to {_csv_path}")
else:
    with open("bench_summary.csv", "w") as _f:
        _f.write(_csv_str)
    print("Summary CSV saved to bench_summary.csv")
